In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# Load feature engineered data
train_fe = pd.read_parquet("../data/processed/train_fe.parquet")
val_fe = pd.read_parquet("../data/processed/val_fe.parquet")

target = "sales"

exclude_cols = [
    "sales", "date", "item_id", "dept_id",
    "cat_id", "store_id", "state_id"
]

features = [col for col in train_fe.columns if col not in exclude_cols]

print("Train FE:", train_fe.shape)

Train FE: (5661993, 21)


In [4]:
train_sample = train_fe.sample(n=300_000, random_state=42)

X_train_s = train_sample[features]
y_train_s = train_sample[target]

print(X_train_s.shape)

(300000, 14)


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.mean(np.where(denom == 0, 0, np.abs(y_true - y_pred) / denom)) * 100

    return rmse, mae, smape

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "num_leaves": [31, 63],
    "max_depth": [-1, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
}

lgb_base = lgb.LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

grid = GridSearchCV(
    estimator=lgb_base,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train_s, y_train_s)

print("Best Params:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best Params: {'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV RMSE: 2.271834177136688


In [8]:
X_val = val_fe[features]
y_val = val_fe[target]

best_lgb = grid.best_estimator_
val_preds = best_lgb.predict(X_val)

rmse, mae, smape = evaluate(y_val, val_preds)

print("Grid Search LightGBM (Validation)")
print("RMSE:", rmse)
print("MAE:", mae)
print("SMAPE:", smape)

Grid Search LightGBM (Validation)
RMSE: 2.059937472271713
MAE: 1.0501471360171934
SMAPE: 134.62242840768


In [9]:
import os, json
os.makedirs("../models", exist_ok=True)

with open("../models/grid_best_params.json", "w") as f:
    json.dump(grid.best_params_, f, indent=2)

print("Saved ../models/grid_best_params.json")

Saved ../models/grid_best_params.json
